In [1]:
import os
os.environ["JAVA_HOME"] = r"C:\Users\hp\AppData\Local\Programs\Eclipse Adoptium\jdk-17.0.18.8-hotspot"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PYSPARK_PYTHON"] = r"C:\Users\hp\anaconda3\envs\pyspark_env\python.exe"

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, regexp_replace, trim

spark = SparkSession.builder \
    .appName("04 - Spark SQL") \
    .config("spark.driver.memory", "2g") \
    .config("spark.ui.enabled", "false") \
    .getOrCreate()

print("✅ Spark Ready!", spark.version)

✅ Spark Ready! 4.1.1


In [3]:
df = spark.read.csv(
    r"C:\Users\hp\BIGDATAPROJECT\data\Tweets.csv",
    header=True, inferSchema=True
)

df_clean = df.select(
    "tweet_id","airline_sentiment","airline_sentiment_confidence",
    "negativereason","airline","retweet_count",
    "text","tweet_created","tweet_location"
).dropna(subset=["tweet_id","airline_sentiment","airline","text"])

df_clean = df_clean.withColumn("clean_text",
    trim(lower(regexp_replace(regexp_replace(
        regexp_replace(col("text"), r"http\S+",""),
    r"@\w+",""),r"[^a-zA-Z\s]","")))
)

# Register SQL view
df_clean.createOrReplaceTempView("tweets")
print("✅ SQL view registered! Rows:", df_clean.count())

✅ SQL view registered! Rows: 14632


In [4]:
print("=== Sentiment Distribution with Percentage ===")
spark.sql("""
    SELECT
        airline_sentiment,
        COUNT(*) AS total,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS percentage
    FROM tweets
    GROUP BY airline_sentiment
    ORDER BY total DESC
""").show()

=== Sentiment Distribution with Percentage ===
+-----------------+-----+----------+
|airline_sentiment|total|percentage|
+-----------------+-----+----------+
|         negative| 9170|     62.67|
|          neutral| 3099|     21.18|
|         positive| 2363|     16.15|
+-----------------+-----+----------+



In [5]:
print("=== Airline Scorecard ===")
spark.sql("""
    SELECT
        airline,
        SUM(CASE WHEN airline_sentiment='positive' THEN 1 ELSE 0 END) AS positive,
        SUM(CASE WHEN airline_sentiment='neutral'  THEN 1 ELSE 0 END) AS neutral,
        SUM(CASE WHEN airline_sentiment='negative' THEN 1 ELSE 0 END) AS negative,
        COUNT(*) AS total,
        ROUND(SUM(CASE WHEN airline_sentiment='positive' THEN 1 ELSE 0 END)
              * 100.0 / COUNT(*), 2) AS positive_pct
    FROM tweets
    GROUP BY airline
    ORDER BY positive_pct DESC
""").show()

=== Airline Scorecard ===
+--------------+--------+-------+--------+-----+------------+
|       airline|positive|neutral|negative|total|positive_pct|
+--------------+--------+-------+--------+-----+------------+
|Virgin America|     152|    171|     181|  504|       30.16|
|         Delta|     544|    723|     953| 2220|       24.50|
|     Southwest|     570|    664|    1185| 2419|       23.56|
|        United|     492|    697|    2630| 3819|       12.88|
|      American|     336|    463|    1958| 2757|       12.19|
|    US Airways|     269|    381|    2263| 2913|        9.23|
+--------------+--------+-------+--------+-----+------------+



In [6]:
print("=== Top Negative Reason per Airline ===")
spark.sql("""
    SELECT airline, negativereason, COUNT(*) AS count
    FROM tweets
    WHERE airline_sentiment = 'negative'
    AND negativereason IS NOT NULL
    GROUP BY airline, negativereason
    ORDER BY airline, count DESC
""").show(30)

=== Top Negative Reason per Airline ===
+---------+--------------------+-----+
|  airline|      negativereason|count|
+---------+--------------------+-----+
| American|Customer Service ...|  768|
| American|         Late Flight|  249|
| American|    Cancelled Flight|  245|
| American|          Can't Tell|  198|
| American|        Lost Luggage|  148|
| American|Flight Booking Pr...|  130|
| American|Flight Attendant ...|   87|
| American|          Bad Flight|   87|
| American|           longlines|   34|
| American|     Damaged Luggage|   12|
|    Delta|         Late Flight|  269|
|    Delta|Customer Service ...|  199|
|    Delta|          Can't Tell|  185|
|    Delta|          Bad Flight|   64|
|    Delta|Flight Attendant ...|   60|
|    Delta|        Lost Luggage|   56|
|    Delta|    Cancelled Flight|   51|
|    Delta|Flight Booking Pr...|   44|
|    Delta|           longlines|   14|
|    Delta|     Damaged Luggage|   11|
|Southwest|Customer Service ...|  390|
|Southwest|    Cancelled

In [7]:
print("=== Tweet Activity by Hour ===")
spark.sql("""
    SELECT
        SUBSTRING(tweet_created, 12, 2) AS hour,
        COUNT(*) AS total_tweets,
        SUM(CASE WHEN airline_sentiment='negative' THEN 1 ELSE 0 END) AS negative
    FROM tweets
    WHERE tweet_created IS NOT NULL
    GROUP BY hour
    ORDER BY hour
""").show(24)

=== Tweet Activity by Hour ===
+----+------------+--------+
|hour|total_tweets|negative|
+----+------------+--------+
|    |           4|       2|
|   b|           1|       1|
|   g|           1|       0|
|  00|         129|      95|
|  01|         110|      79|
|  02|         174|     109|
|  03|         221|     145|
|  04|         363|     234|
|  05|         468|     305|
|  06|         609|     387|
|  07|         765|     467|
|  08|         912|     552|
|  09|         993|     609|
|  10|         948|     535|
|  11|         974|     567|
|  12|         815|     480|
|  13|         885|     548|
|  14|         916|     578|
|  15|         780|     495|
|  16|         719|     454|
|  17|         747|     449|
|  18|         742|     469|
|  19|         667|     477|
|  20|         597|     384|
+----+------------+--------+
only showing top 24 rows


In [8]:
import time
from pymongo import MongoClient

# Spark SQL timing
start = time.time()
spark.sql("""
    SELECT airline_sentiment, COUNT(*) as count
    FROM tweets
    GROUP BY airline_sentiment
""").collect()
spark_time = time.time() - start
print(f"⚡ Spark SQL time:    {spark_time:.4f} sec")

# MongoDB timing
client = MongoClient("mongodb://localhost:27017/")
collection = client["tweets_db"]["airline_tweets"]

start = time.time()
list(collection.aggregate([
    {"$group": {"_id": "$airline_sentiment", "count": {"$sum": 1}}}
]))
mongo_time = time.time() - start
print(f"🍃 MongoDB time:      {mongo_time:.4f} sec")

# Winner
print(f"\n{'⚡ Spark SQL' if spark_time < mongo_time else '🍃 MongoDB'} was faster!")
print(f"Difference: {abs(spark_time - mongo_time):.4f} sec")

⚡ Spark SQL time:    0.3608 sec
🍃 MongoDB time:      0.0598 sec

🍃 MongoDB was faster!
Difference: 0.3010 sec


In [1]:
import tempfile
import os

# This shows your exact temp folder path
print("Temp folder:", tempfile.gettempdir())

Temp folder: C:\Users\hp\AppData\Local\Temp


In [ ]:
# NOTE: Parquet format skipped due to Windows + Hadoop
# native library limitation on local development environment.
# In a Linux/cloud environment, Parquet provides:
# - 3-5x faster read speeds than CSV
# - 60-70% smaller file size
# - Better columnar compression
import time

# Try saving to temp directory
parquet_path = "file:///C:/Users/hp/AppData/Local/Temp/tweets_parquet"

try:
    df_clean.coalesce(1).write.parquet(parquet_path, mode="overwrite")
    print("✅ Parquet saved!")

    # Read back and time it
    start = time.time()
    spark.read.parquet(parquet_path).count()
    parquet_time = time.time() - start

    start = time.time()
    spark.read.csv(
        r"C:\Users\hp\BIGDATAPROJECT\data\Tweets.csv",
        header=True, inferSchema=True
    ).count()
    csv_time = time.time() - start

    print(f"📄 CSV time:     {csv_time:.4f} sec")
    print(f"📦 Parquet time: {parquet_time:.4f} sec")
    print(f"{'📦 Parquet' if parquet_time < csv_time else '📄 CSV'} was faster!")

except Exception as e:
    print("⚠️ Parquet not supported in this setup")
    print("This is a known Windows + Hadoop limitation")
    print("For your report: note that Parquet requires full Hadoop setup on Windows")

⚠️ Parquet not supported in this setup
This is a known Windows + Hadoop limitation
For your report: note that Parquet requires full Hadoop setup on Windows
